In [ ]:
import pandas as pd

train = pd.read_parquet('dataset/train.parquet')
val = pd.read_parquet('dataset/val.parquet')
test = pd.read_parquet('dataset/test.parquet')
test_unseen = pd.read_parquet('dataset/test_unseen.parquet')

print("train筆數：", len(train))
print("val筆數：", len(val))
print("test筆數：", len(test))
print("test_unseen筆數：", len(test_unseen))

print("\n欄位清單：", train.columns.tolist())
print("\ntrain前3筆：")
print(train.head(3))

In [ ]:
# 把seen建築的train/val/test合併回一張完整表
seen_full = pd.concat([train, val, test], ignore_index=True)
seen_full = seen_full.sort_values(['building_id', 'timestamp']).reset_index(drop=True)

# 每棟建築各自建立從0開始的連續time_idx
seen_full['time_idx'] = seen_full.groupby('building_id').cumcount()

print("seen_full筆數：", len(seen_full))
print("time_idx範圍：", seen_full['time_idx'].min(), "~", seen_full['time_idx'].max())

# 檢查每棟建築的time_idx是否連續無跳號
gap_check = seen_full.groupby('building_id')['time_idx'].apply(lambda x: (x.diff().dropna() == 1).all())
print("\n是否所有建築的time_idx都連續無跳號：", gap_check.all())
print("有跳號問題的建築數：", (~gap_check).sum())

# test_unseen同樣處理
test_unseen_full = test_unseen.sort_values(['building_id', 'timestamp']).reset_index(drop=True)
test_unseen_full['time_idx'] = test_unseen_full.groupby('building_id').cumcount()

print("\ntest_unseen_full筆數：", len(test_unseen_full))

In [ ]:
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer

# 定義預測窗口設定
max_encoder_length = 168  # 用過去168小時（一週）當作輸入視窗
max_prediction_length = 24  # 預測未來24小時

# 確保類別型欄位是字串型態
seen_full['building_id'] = seen_full['building_id'].astype(str)
seen_full['site_id'] = seen_full['site_id'].astype(str)
seen_full['hour'] = seen_full['hour'].astype(str)
seen_full['weekday'] = seen_full['weekday'].astype(str)
seen_full['is_weekend'] = seen_full['is_weekend'].astype(str)
seen_full['month'] = seen_full['month'].astype(str)

# 訓練集的time_idx上限：用train集最後的time_idx當切點
training_cutoff = seen_full[seen_full['timestamp'] < '2017-06-01']['time_idx'].max()

print("training_cutoff（train集結束的time_idx）：", training_cutoff)

training_dataset = TimeSeriesDataSet(
    seen_full[lambda x: x.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="load_intensity",
    group_ids=["building_id"],
    min_encoder_length=max_encoder_length // 2,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,
    
    static_categoricals=["building_id", "site_id"],
    static_reals=["sqm"],
    
    time_varying_known_categoricals=["hour", "weekday", "is_weekend", "month"],
    time_varying_known_reals=["time_idx", "airTemperature", "dewTemperature", "windSpeed", "windDirection"],
    
    time_varying_unknown_reals=["load_intensity"],
    
    target_normalizer=GroupNormalizer(groups=["building_id"], transformation="softplus"),
    
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

print("\n訓練資料集建立完成，樣本數：", len(training_dataset))

In [ ]:
# 用training_dataset的設定，從完整seen_full資料建立validation dataset
validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset, seen_full, predict=True, stop_randomization=True
)

# 建立DataLoader
# 本機有獨立GPU且不受Colab連線限制，可以考慮把batch_size開大一點加速訓練
# num_workers也可以視你的CPU核心數調整（Windows上有時候num_workers>0會有多進程問題，先從0開始比較保險）
batch_size = 512
train_dataloader = training_dataset.to_dataloader(
    train=True, batch_size=512, num_workers=4,
    persistent_workers=True, pin_memory=True,
)
val_dataloader = validation_dataset.to_dataloader(
    train=False, batch_size=1024, num_workers=2,
    persistent_workers=True, pin_memory=True,
)

print("train batch數：", len(train_dataloader))
print("val batch數：", len(val_dataloader))

===========重建TFT模型=============

In [ ]:
import torch
import lightning.pytorch as pl
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')

quantiles = [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]

tft = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=0.03,
    hidden_size=32,
    attention_head_size=1,
    dropout=0.1,
    hidden_continuous_size=16,
    loss=QuantileLoss(quantiles=quantiles),
    log_interval=10,
    optimizer="Adam",
    reduce_on_plateau_patience=4,
)

if torch.cuda.is_available():
    tft = tft.cuda()

print(f"模型參數量：{tft.size()/1e3:.1f}k")
print("模型參數所在裝置：", next(tft.parameters()).device)

In [ ]:
!pip install tensorboard

In [ ]:
import subprocess
result = subprocess.run(['pip', 'show', 'tensorboard'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger

early_stop_callback = EarlyStopping(
    monitor="val_loss", min_delta=1e-4, patience=3, verbose=True, mode="min"
)
lr_logger = LearningRateMonitor()
checkpoint_callback = ModelCheckpoint(
    dirpath="tft_checkpoints",
    filename="best-tft-{epoch:02d}-{val_loss:.4f}",
    monitor="val_loss",
    mode="min",
    save_top_k=1,
)
logger = TensorBoardLogger("tft_logs", name="tft_office_bdg2")

trainer = pl.Trainer(
    max_epochs=15,
    accelerator="gpu",
    devices=1,
    enable_model_summary=True,
    gradient_clip_val=0.1,
    callbacks=[lr_logger, early_stop_callback, checkpoint_callback],
    logger=logger,
    precision="bf16-mixed"
)

print("\n開始正式訓練（batch_size=512）...")

trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)

print("\n訓練完成")
print("最佳模型儲存位置：", checkpoint_callback.best_model_path)
print("最佳val_loss：", checkpoint_callback.best_model_score.item())

=================================================================

In [ ]:
print(training_dataset._categorical_encoders["building_id"])
print("add_nan設定:", training_dataset._categorical_encoders["building_id"].add_nan)

In [ ]:
# 找到building_id的embedding層（pytorch_forecasting內建模型共用MultiEmbedding結構）
building_embed_layer = tft.input_embeddings.embeddings["building_id"]
print("building_id embedding形狀:", building_embed_layer.weight.shape)  # 應該是 (251, embedding_dim)

import torch
mean_embedding = building_embed_layer.weight.data.mean(dim=0, keepdim=True)
print("平均向量形狀:", mean_embedding.shape)

In [ ]:
from copy import deepcopy

# 1. 把embedding矩陣擴充一列（index 251 = 平均向量）
old_weight = building_embed_layer.weight.data  # (251, 32)
new_weight = torch.cat([old_weight, mean_embedding], dim=0)  # (252, 32)

new_embed_layer = torch.nn.Embedding(252, 32)
new_embed_layer.weight.data = new_weight
tft.input_embeddings.embeddings["building_id"] = new_embed_layer

# 2. 複製一份building_id的類別編碼器，手動把test_unseen的28棟建築都指向index 251
patched_encoder = deepcopy(training_dataset._categorical_encoders["building_id"])
unseen_building_ids = test_unseen_full["building_id"].unique()
n_new = 0
for bid in unseen_building_ids:
    if bid not in patched_encoder.classes_:
        patched_encoder.classes_[bid] = 251
        n_new += 1
print(f"新增了{n_new}個building_id對應到index 251（應該是28）")

In [ ]:
test_unseen_full["building_id"] = test_unseen_full["building_id"].astype(str)
test_unseen_full["site_id"] = test_unseen_full["site_id"].astype(str)
test_unseen_full["hour"] = test_unseen_full["hour"].astype(str)
test_unseen_full["weekday"] = test_unseen_full["weekday"].astype(str)
test_unseen_full["is_weekend"] = test_unseen_full["is_weekend"].astype(str)
test_unseen_full["month"] = test_unseen_full["month"].astype(str)

## test seen

In [ ]:
import pandas as pd
import torch
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer

# 1. 重新載入資料，組回seen_full
train = pd.read_parquet("dataset/train.parquet")
val = pd.read_parquet("dataset/val.parquet")
test = pd.read_parquet("dataset/test.parquet")
seen_full = pd.concat([train, val, test], ignore_index=True)
seen_full['timestamp'] = pd.to_datetime(seen_full['timestamp'])
seen_full = seen_full.sort_values(['building_id', 'timestamp']).reset_index(drop=True)
seen_full['time_idx'] = (
    (seen_full['timestamp'] - seen_full['timestamp'].min())
    .dt.total_seconds().div(3600).astype(int)
)
print("time_idx範圍：", seen_full['time_idx'].min(), "~", seen_full['time_idx'].max())

# 2. 確保類別型欄位是字串型態
for col in ["building_id", "site_id", "hour", "weekday", "is_weekend", "month"]:
    seen_full[col] = seen_full[col].astype(str)

# 3. 建立training_dataset（跟原本TFT.ipynb一致）
max_encoder_length = 168
max_prediction_length = 24

training_cutoff = seen_full[seen_full['timestamp'] < '2017-06-01']['time_idx'].max()
print("training_cutoff：", training_cutoff)

training_dataset = TimeSeriesDataSet(
    seen_full[lambda x: x.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="load_intensity",
    group_ids=["building_id"],
    min_encoder_length=max_encoder_length // 2,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,
    static_categoricals=["building_id", "site_id"],
    static_reals=["sqm"],
    time_varying_known_categoricals=["hour", "weekday", "is_weekend", "month"],
    time_varying_known_reals=["time_idx", "airTemperature", "dewTemperature", "windSpeed", "windDirection"],
    time_varying_unknown_reals=["load_intensity"],
    target_normalizer=GroupNormalizer(groups=["building_id"], transformation="softplus"),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

# 4. 從checkpoint載入訓練好的TFT模型
tft = TemporalFusionTransformer.load_from_checkpoint(
    r"C:\Users\peggy\Desktop\能源\tft_checkpoints\best-tft-epoch=02-val_loss=0.0074.ckpt"
)
tft.eval()
if torch.cuda.is_available():
    tft = tft.cuda()

# 5. 建立test_dataset / test_dataloader（限定decoder窗口落在test期間）
test_start_time_idx = seen_full[seen_full['timestamp'] >= '2017-10-01']['time_idx'].min()
print("test期間開始的time_idx：", test_start_time_idx)

test_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    seen_full,
    predict=False,
    stop_randomization=True,
    min_prediction_idx=test_start_time_idx,
    min_prediction_length=max_prediction_length,  # 強制固定24小時，捨棄尾端不足24小時的樣本
)
print("test_dataset樣本數：", len(test_dataset))

test_dataloader = test_dataset.to_dataloader(train=False, batch_size=128, num_workers=0)

predictions = tft.predict(test_dataloader, mode="prediction", return_y=True)
y_pred = predictions.output.cpu()
y_true = predictions.y[0].cpu()

print("y_pred有NaN:", torch.isnan(y_pred).any().item())
print("y_true有NaN:", torch.isnan(y_true).any().item())

rmse = torch.sqrt(torch.mean((y_pred - y_true) ** 2)).item()
mae = torch.mean(torch.abs(y_pred - y_true)).item()
print(f"TFT — test set RMSE: {rmse:.4f}, MAE: {mae:.4f}")

## test unseen

In [ ]:
test_unseen_full = pd.read_parquet("dataset/test_unseen.parquet")

test_unseen_full["timestamp"] = pd.to_datetime(test_unseen_full["timestamp"])
test_unseen_full = test_unseen_full.sort_values(["building_id", "timestamp"]).reset_index(drop=True)
test_unseen_full["time_idx"] = (
    (test_unseen_full["timestamp"] - test_unseen_full["timestamp"].min())
    .dt.total_seconds().div(3600).astype(int)
)

for col in ["building_id", "site_id", "hour", "weekday", "is_weekend", "month"]:
    test_unseen_full[col] = test_unseen_full[col].astype(str)

print("test_unseen_full筆數：", len(test_unseen_full))
print("building_id數量：", test_unseen_full["building_id"].nunique())

In [ ]:
from copy import deepcopy

# 1. 取得building_id的embedding層，算出251棟已知建築的平均向量
building_embed_layer = tft.input_embeddings.embeddings["building_id"]
print("building_id embedding形狀:", building_embed_layer.weight.shape)

mean_embedding = building_embed_layer.weight.data.mean(dim=0, keepdim=True)

# 2. 把embedding矩陣擴充一列（index 251 = 平均向量）
old_weight = building_embed_layer.weight.data
new_weight = torch.cat([old_weight, mean_embedding], dim=0)

new_embed_layer = torch.nn.Embedding(new_weight.shape[0], new_weight.shape[1])
new_embed_layer.weight.data = new_weight
if torch.cuda.is_available():
    new_embed_layer = new_embed_layer.cuda()
tft.input_embeddings.embeddings["building_id"] = new_embed_layer

# 3. 複製一份building_id的類別編碼器，手動把test_unseen的28棟建築都指向index 251
patched_encoder = deepcopy(training_dataset._categorical_encoders["building_id"])
unseen_building_ids = test_unseen_full["building_id"].unique()
n_new = 0
for bid in unseen_building_ids:
    if bid not in patched_encoder.classes_:
        patched_encoder.classes_[bid] = 251
        n_new += 1
print(f"新增了{n_new}個building_id對應到index 251（應該是28）")

In [ ]:
test_unseen_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    test_unseen_full,
    predict=False,
    stop_randomization=True,
    categorical_encoders={"building_id": patched_encoder},
    min_prediction_length=max_prediction_length,
)
print("test_unseen_dataset樣本數：", len(test_unseen_dataset))

test_unseen_dataloader = test_unseen_dataset.to_dataloader(train=False, batch_size=128, num_workers=0)

predictions_unseen = tft.predict(test_unseen_dataloader, mode="prediction", return_y=True, return_x=True)
y_pred_unseen = predictions_unseen.output.cpu()
y_true_unseen = predictions_unseen.y[0].cpu()
encoder_lengths = predictions_unseen.x["encoder_lengths"].cpu()

print("y_pred_unseen有NaN:", torch.isnan(y_pred_unseen).any().item(),
      "數量:", torch.isnan(y_pred_unseen).sum().item(), "/", y_pred_unseen.numel())

nan_mask = torch.isnan(y_pred_unseen).any(dim=1)
print("NaN樣本數：", nan_mask.sum().item())
if nan_mask.sum() > 0:
    print("NaN樣本的encoder_length: min=", encoder_lengths[nan_mask].min().item(),
          "max=", encoder_lengths[nan_mask].max().item())

In [ ]:
rmse_unseen = torch.sqrt(torch.mean((y_pred_unseen - y_true_unseen) ** 2)).item()
mae_unseen = torch.mean(torch.abs(y_pred_unseen - y_true_unseen)).item()
print(f"TFT — test_unseen set RMSE: {rmse_unseen:.4f}, MAE: {mae_unseen:.4f}")

### CRPS用TFT的分位數預測（7個分位數：0.05/0.1/0.25/0.5/0.75/0.9/0.95）

In [ ]:
quantiles = [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]

def compute_crps(model, dataloader):
    preds_q = model.predict(dataloader, mode="quantiles", return_y=True)
    y_pred_q = preds_q.output.cpu()   # shape: (n_samples, 24, 7)
    y_true = preds_q.y[0].cpu()       # shape: (n_samples, 24)

    y_true_expanded = y_true.unsqueeze(-1)  # (n_samples, 24, 1)
    pinball_losses = []
    for i, q in enumerate(quantiles):
        diff = y_true_expanded[..., 0] - y_pred_q[..., i]
        pinball = torch.max(q * diff, (q - 1) * diff)
        pinball_losses.append(pinball)

    pinball_stack = torch.stack(pinball_losses, dim=-1)  # (n_samples, 24, 7)
    crps = 2 * pinball_stack.mean().item()
    return crps

crps_test = compute_crps(tft, test_dataloader)
print(f"TFT — test set CRPS: {crps_test:.4f}")

crps_test_unseen = compute_crps(tft, test_unseen_dataloader)
print(f"TFT — test_unseen set CRPS: {crps_test_unseen:.4f}")

## test_seen 的校準檢查

In [ ]:
quantiles = [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]
q10_idx = quantiles.index(0.1)
q90_idx = quantiles.index(0.9)

# 取得test集的分位數預測
preds_q_test = tft.predict(test_dataloader, mode="quantiles", return_y=True)
y_pred_q_test = preds_q_test.output.cpu()   # (n_samples, 24, 7)
y_true_test = preds_q_test.y[0].cpu()       # (n_samples, 24)

q10_test = y_pred_q_test[..., q10_idx]
q90_test = y_pred_q_test[..., q90_idx]

# 規則一核心邏輯：偏高或偏低
anomaly_high_test = y_true_test > q90_test
anomaly_low_test = y_true_test < q10_test
anomaly_rule1_test = anomaly_high_test | anomaly_low_test

# 嚴重度（偏高、偏低分開算）
severity_test = torch.zeros_like(y_true_test)
severity_test[anomaly_high_test] = (
    (y_true_test[anomaly_high_test] - q90_test[anomaly_high_test])
    / (q90_test[anomaly_high_test] - q10_test[anomaly_high_test])
)
severity_test[anomaly_low_test] = (
    (q10_test[anomaly_low_test] - y_true_test[anomaly_low_test])
    / (q90_test[anomaly_low_test] - q10_test[anomaly_low_test])
)

print("=== test集 規則一結果 ===")
print("總異常比例（偏高+偏低）:", anomaly_rule1_test.float().mean().item())
print("偏高比例:", anomaly_high_test.float().mean().item())
print("偏低比例:", anomaly_low_test.float().mean().item())

coverage_test = 1 - anomaly_rule1_test.float().mean().item()
print(f"實際區間覆蓋率: {coverage_test*100:.2f}%（80%區間，理論值應約80%）")

## test_unseen 的校準檢查

In [ ]:
preds_q_unseen = tft.predict(test_unseen_dataloader, mode="quantiles", return_y=True)
y_pred_q_unseen = preds_q_unseen.output.cpu()
y_true_unseen = preds_q_unseen.y[0].cpu()

q10_unseen = y_pred_q_unseen[..., q10_idx]
q90_unseen = y_pred_q_unseen[..., q90_idx]

anomaly_high_unseen = y_true_unseen > q90_unseen
anomaly_low_unseen = y_true_unseen < q10_unseen
anomaly_rule1_unseen = anomaly_high_unseen | anomaly_low_unseen

severity_unseen = torch.zeros_like(y_true_unseen)
severity_unseen[anomaly_high_unseen] = (
    (y_true_unseen[anomaly_high_unseen] - q90_unseen[anomaly_high_unseen])
    / (q90_unseen[anomaly_high_unseen] - q10_unseen[anomaly_high_unseen])
)
severity_unseen[anomaly_low_unseen] = (
    (q10_unseen[anomaly_low_unseen] - y_true_unseen[anomaly_low_unseen])
    / (q90_unseen[anomaly_low_unseen] - q10_unseen[anomaly_low_unseen])
)

print("=== test_unseen集 規則一結果 ===")
print("總異常比例（偏高+偏低）:", anomaly_rule1_unseen.float().mean().item())
print("偏高比例:", anomaly_high_unseen.float().mean().item())
print("偏低比例:", anomaly_low_unseen.float().mean().item())

coverage_unseen = 1 - anomaly_rule1_unseen.float().mean().item()
print(f"實際區間覆蓋率: {coverage_unseen*100:.2f}%（80%區間，理論值應約80%）")

第一步：嚴重度分布檢查

In [ ]:
severity_flagged_test = severity_test[anomaly_rule1_test]
print("test集規則一異常的嚴重度分布：")
print("最小值:", severity_flagged_test.min().item())
print("中位數:", severity_flagged_test.median().item())
print("平均值:", severity_flagged_test.mean().item())
print("90百分位:", severity_flagged_test.quantile(0.90).item())
print("99百分位:", severity_flagged_test.quantile(0.99).item())
print("最大值:", severity_flagged_test.max().item())

# 看嚴重度分布是不是「多數輕度、少數極端」的合理形狀
import numpy as np
percentiles = [10, 25, 50, 75, 90, 95, 99]
for p in percentiles:
    print(f"{p}百分位: {severity_flagged_test.quantile(p/100).item():.4f}")

In [ ]:
interval_width_test = q90_test - q10_test
print("區間寬度分布：")
print("最小值:", interval_width_test.min().item())
for p in [0.1, 1, 5, 10, 25, 50]:
    print(f"{p}百分位: {interval_width_test.quantile(p/100).item():.6f}")

# 看嚴重度最誇張的那些樣本，它們的區間寬度是不是特別小
top_severity_idx = severity_test.flatten().argsort(descending=True)[:20]
flat_width = interval_width_test.flatten()
print("\n嚴重度最高的20個樣本，對應的區間寬度：")
print(flat_width[top_severity_idx])

In [ ]:
# 以 test 集區間寬度分布的第5百分位數作為下限（避免退化區間造成除法爆炸）
epsilon_floor = interval_width_test.quantile(0.05).item()
print("採用的區間寬度下限（第5百分位數）:", epsilon_floor)
print("受此下限影響的樣本比例（test）:", (interval_width_test < epsilon_floor).float().mean().item())

# 重新計算嚴重度（分母套用下限，其餘邏輯不變）
width_floored_test = torch.clamp(interval_width_test, min=epsilon_floor)
severity_high_test = (y_true_test - q90_test) / width_floored_test
severity_low_test  = (q10_test - y_true_test) / width_floored_test
severity_test = torch.where(
    y_true_test > q90_test, severity_high_test,
    torch.where(y_true_test < q10_test, severity_low_test, torch.zeros_like(y_true_test))
)

severity_flagged_test = severity_test[anomaly_rule1_test]
print("\n修正後的嚴重度分布（test集）：")
print("最小值:", severity_flagged_test.min().item())
print("最大值:", severity_flagged_test.max().item())
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f"{p}百分位: {severity_flagged_test.quantile(p/100).item():.4f}")

第二步：校準「驟降型」合成注入強度

In [ ]:
# 用train.parquet（或seen_full）算每棟建築的「低於自己中位數」比例分布
low_ratios = seen_full.groupby('building_id')['load_intensity'].apply(
    lambda s: (s / s.median())
)
# 篩選出比中位數低的部分，看這些比例的低百分位數
low_ratios_below_median = low_ratios[low_ratios < 1]
print("低於中位數的ratio分布：")
for p in [50, 25, 10, 5, 1, 0.1]:
    print(f"{p}百分位: {low_ratios_below_median.quantile(p/100):.4f}")

In [ ]:
interval_width_unseen = q90_unseen - q10_unseen

width_floored_unseen = torch.clamp(interval_width_unseen, min=epsilon_floor)
severity_high_unseen = (y_true_unseen - q90_unseen) / width_floored_unseen
severity_low_unseen  = (q10_unseen - y_true_unseen) / width_floored_unseen
severity_unseen = torch.where(
    y_true_unseen > q90_unseen, severity_high_unseen,
    torch.where(y_true_unseen < q10_unseen, severity_low_unseen, torch.zeros_like(y_true_unseen))
)

severity_flagged_unseen = severity_unseen[anomaly_rule1_unseen]
print("修正後的嚴重度分布（test_unseen集）：")
print("最大值:", severity_flagged_unseen.max().item())
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f"{p}百分位: {severity_flagged_unseen.quantile(p/100).item():.4f}")

In [ ]:
low_ratios = seen_full.groupby('building_id')['load_intensity'].apply(
    lambda s: (s / s.median())
)
low_ratios_below_median = low_ratios[low_ratios < 1]
print("低於中位數的ratio分布：")
for p in [50, 25, 10, 5, 1, 0.1]:
    print(f"{p}百分位: {low_ratios_below_median.quantile(p/100):.4f}")

In [ ]:
torch.manual_seed(42)
y_true_flat = y_true_test.flatten()
q10_flat = q10_test.flatten()
q90_flat = q90_test.flatten()

n_total = y_true_flat.numel()
sample_size = 5000
sample_idx = torch.randperm(n_total)[:sample_size]

drop_ratios = [1.0, 0.9, 0.7, 0.5, 0.3, 0.2, 0.1, 0.05]
print("驟降型合成注入偵測率（不同注入比例）：")
for ratio in drop_ratios:
    y_injected = y_true_flat[sample_idx] * ratio
    detected = (y_injected < q10_flat[sample_idx]).float().mean().item()
    print(f"保留原值{ratio*100:.0f}%（驟降{(1-ratio)*100:.0f}%）→ 偵測率: {detected:.4f}")

測試

In [ ]:
# 1. 重新取得test集預測，加上 return_index=True 以取得每個樣本對應的 building_id 與 time_idx
raw_predictions_test = tft.predict(
    test_dataloader,
    mode="quantiles",
    return_y=True,
    return_index=True,
)
index_df = raw_predictions_test.index
q_test = raw_predictions_test.output
y_test_2d = raw_predictions_test.y[0]

q10_test_2d = q_test[..., 1]   # 分位數順序 0.05,0.1,0.25,0.5,0.75,0.9,0.95 → 索引1是q10
q90_test_2d = q_test[..., 5]   # 索引5是q90

# 2. 套用先前的epsilon_floor，計算嚴重度（保留樣本x小時的2D結構，不要攤平）
width_2d = q90_test_2d - q10_test_2d
width_floored_2d = torch.clamp(width_2d, min=epsilon_floor)
anomaly_high_2d = y_test_2d > q90_test_2d
anomaly_low_2d  = y_test_2d < q10_test_2d
severity_2d = torch.where(
    anomaly_high_2d, (y_test_2d - q90_test_2d) / width_floored_2d,
    torch.where(anomaly_low_2d, (q10_test_2d - y_test_2d) / width_floored_2d, torch.zeros_like(y_test_2d))
)
anomaly_2d = anomaly_high_2d | anomaly_low_2d

# 3. 挑嚴重度最接近「異常樣本中位數」的那一筆，當作代表性案例
median_severity = severity_2d[anomaly_2d].median().item()
diff = torch.where(anomaly_2d, (severity_2d - median_severity).abs(), torch.full_like(severity_2d, float("inf")))
sample_idx, hour_idx = (diff == diff.min()).nonzero()[0].tolist()

decoder_start_time_idx = index_df.iloc[sample_idx]["time_idx"]
case_time_idx = decoder_start_time_idx + hour_idx
base_timestamp = seen_full["timestamp"].min()
case_timestamp = base_timestamp + pd.Timedelta(hours=case_time_idx)

building_id_case = index_df.iloc[sample_idx]["building_id"]
building_row = seen_full[seen_full["building_id"] == building_id_case].iloc[0]

case_rule1 = {
    "id": "R1-demo",
    "type": "interval",
    "building_id": building_id_case,
    "site_id": building_row["site_id"],
    "sqm": float(building_row["sqm"]),
    "timestamp": str(case_timestamp),
    "y_true": y_test_2d[sample_idx, hour_idx].item(),
    "q10": q10_test_2d[sample_idx, hour_idx].item(),
    "q90": q90_test_2d[sample_idx, hour_idx].item(),
    "severity": severity_2d[sample_idx, hour_idx].item(),
    "direction": "high" if anomaly_high_2d[sample_idx, hour_idx].item() else "low",
}
print(case_rule1)